# Craigslist listing price — results and interpretation

## Project summary (grading)

- **Task:** predict **used-car listing price** (USD) from enriched Craigslist rows (`regex` + **LLM** ETL → master CSV in GCS, materialized as `listings_master_llm.csv`).
- **Modeling:** **time-aware** split (train on all local calendar days before the latest day; **holdout = latest day**). No random split is used for the main grade-facing evaluation.
- **This notebook:** reads **synced files under `results/` only** — **no retraining**, **no GCP credentials**. Reproduce by running `train-dt` (not `dry_run`) and the **Sync model artifacts** workflow.

**Holdout metrics** (MAE, MAPE, RMSE, Bias) are always in **original dollars** (`actual_price` vs `pred_price`). If training used `log1p(price)`, predictions are **back-transformed with `expm1`** before metrics and `predictions.csv`.

## 1. Data and features (high level)

- **Source:** Craigslist car listings scraped to GCS; **regex** structured extract + **Vertex Gemini** enrichment; master table **`listings_master_llm.csv`**.
- **Key enriched fields** include make/model, mileage, year, transmission, fuel, drive, condition, title status, body type, seller type, location (city/state), and **`zip_prefix`** from a cleaned 5-digit ZIP (invalid ZIPs nulled).
- **Training filters** (before fitting): positive price in range, plausible model year vs scrape year, mileage cap (see **`max_mileage_cap`** in **`filtering`**), drop non-standard **title** rows (salvage / parts-only / missing) and **condition = project**. Counts are in **`metrics.json` → `filtering`**.
- **Numeric feature variants:** the trainer picks **A**, **B**, or **C** using a fixed **RandomForest + log target** probe on the **calendar validation day**: **A** = `vehicle_age` + `log_mileage` (+ cylinders); **B** = `year_num` + `mileage_num` (+ cylinders); **C** = full numeric set (may include redundant pairs if validation improves). See **`benchmark.feature_variant`** in **`metrics.json`**.

## 2. Modeling strategy

- **Time-aware split:** `scraped_at` → **`America/New_York` local date**. **Train** = all days **before** the latest day; **holdout** = **latest** day only (main evaluation).
- **Calendar validation:** if **≥3** distinct local dates, the **second-latest** day benchmarks every **(model × target)** pair on the **same** rows so comparison is fair and forward-looking.
- **Candidates:** `DecisionTreeRegressor`, `RandomForestRegressor`, `ExtraTreesRegressor`, `HistGradientBoostingRegressor`.
- **Targets:** **`price_num` (raw)** vs **`log1p(price_num)`**. Validation and holdout errors for grading are in **USD** after **`expm1`** when the model was trained on log price.
- **Ranking:** default-params results are ordered by **`val_composite` = MAE + w_rmse×RMSE + w_bias×|bias|`** (see **`metrics.json` → `benchmark.validation_ranking`**). **MAPE** is shown but **not** used to pick the winner (it is unstable when many cheap cars appear).
- **Tuning:** the **top two** **(model, target)** pairs by that ranking are tuned with **`ParameterSampler`** on the same validation day; the **winner** is refit on **all** pre-holdout rows, then scored once on the **latest** holdout.

**Why not random K-fold?** Listing mix and price levels shift by day; random splits can leak future-like patterns. Calendar validation + holdout better match deployment.


In [1]:
# Repo root: Colab shallow-clone, or local checkout with ./results
import os
import sys

DEFAULT_REPO = "https://github.com/OPIM5512-mjb24001/myscrapers-mjb24001-v3.git"
REPO_URL = os.environ.get("NOTEBOOK_REPO_URL", DEFAULT_REPO)
BRANCH = os.environ.get("NOTEBOOK_REPO_BRANCH", "main")

if "google.colab" in sys.modules:
    import subprocess

    subprocess.run(["rm", "-rf", "repo"], check=False)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, "repo"],
        check=True,
    )
    os.chdir("repo")
    print(f"Cloned: {REPO_URL} (branch {BRANCH})")
else:
    if os.path.isdir("results"):
        print("Using current directory as repo root (./results found).")
    elif os.path.isdir(os.path.join("..", "results")):
        os.chdir("..")
        print("Using parent directory as repo root.")
    else:
        print(
            "Tip: run from the repository root so ./results exists, or use Colab to clone."
        )

Using parent directory as repo root.


In [2]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    from IPython.display import display, Image, Markdown
except ImportError:
    display = print
    Image = None

    def Markdown(x):
        print(x)


ROOT = Path(".").resolve()
RESULTS = ROOT / "results"
METRICS_DIR = RESULTS / "metrics"


def latest_run_id() -> str | None:
    if not METRICS_DIR.is_dir():
        return None
    files = sorted(METRICS_DIR.glob("*-metrics.json"))
    if not files:
        return None
    return files[-1].stem.replace("-metrics", "")


run_id = latest_run_id()
metrics_path = METRICS_DIR / f"{run_id}-metrics.json" if run_id else None
model_info_path = METRICS_DIR / f"{run_id}-model_info.json" if run_id else None
bench_csv_path = METRICS_DIR / f"{run_id}-model_benchmark.csv" if run_id else None

metrics = {}
if metrics_path and metrics_path.is_file():
    try:
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"Could not parse metrics JSON: {e}")
        metrics = {}

if run_id:
    display(Markdown(f"**Latest synced run_id:** `{run_id}`"))
else:
    display(
        Markdown(
            "_No `results/metrics/*-metrics.json` found. Train (not dry_run) and run **Sync model artifacts**."
        )
    )

**Latest synced run_id:** `2026040419`

## 3. Model comparison (validation day)

The table below uses **`model_benchmark.csv`** when synced (flat rows: default benchmark + tuned finalists). **`selected=yes`** marks the configuration that was refit on all pre-holdout data. If the CSV is missing (older runs), we fall back to **`metrics.json` → `benchmark`**.

In [3]:
import math

bench_df = pd.DataFrame()


def _composite_row(mae, rmse, bias):
    """Match train-dt val_composite when CSV column missing (older syncs)."""
    try:
        m = float(mae)
    except (TypeError, ValueError):
        return float("nan")
    if not math.isfinite(m):
        return float("nan")
    try:
        r = float(rmse)
        if not math.isfinite(r):
            r = 0.0
    except (TypeError, ValueError):
        r = 0.0
    try:
        b = float(bias)
        if not math.isfinite(b):
            b = 0.0
    except (TypeError, ValueError):
        b = 0.0
    return m + 0.0025 * r + 0.12 * abs(b)


if bench_csv_path and bench_csv_path.is_file():
    try:
        bench_df = pd.read_csv(bench_csv_path)
    except Exception as e:
        print(f"Could not read model_benchmark.csv: {e}")

if bench_df.empty and metrics.get("benchmark"):
    b = metrics["benchmark"]
    rows = []
    for r in b.get("candidate_results") or []:
        rows.append(
            {
                "stage": "benchmark_default_params",
                "model": r.get("model"),
                "target_strategy": r.get("target_strategy"),
                "val_mae": r.get("val_mae"),
                "val_rmse": r.get("val_rmse"),
                "val_mape": r.get("val_mape"),
                "val_bias": r.get("val_bias"),
                "selected": "no",
            }
        )
    chosen = metrics.get("chosen_model")
    ts = metrics.get("target_strategy")
    for f in b.get("tuned_finalists") or []:
        m = f.get("validation_after_tune") or {}
        win = f.get("model") == chosen and f.get("target_strategy") == ts
        rows.append(
            {
                "stage": "tuned_finalist",
                "model": f.get("model"),
                "target_strategy": f.get("target_strategy"),
                "val_mae": m.get("val_mae"),
                "val_rmse": m.get("val_rmse"),
                "val_mape": m.get("val_mape"),
                "val_bias": m.get("val_bias"),
                "selected": "yes" if win else "no",
            }
        )
    bench_df = pd.DataFrame(rows)

if not bench_df.empty and "val_composite" not in bench_df.columns:
    bench_df["val_composite"] = bench_df.apply(
        lambda r: _composite_row(r.get("val_mae"), r.get("val_rmse"), r.get("val_bias")),
        axis=1,
    )

vr = (metrics.get("benchmark") or {}).get("validation_ranking") or {}
if vr:
    display(
        Markdown(
            f"**Validation ranking:** `{vr.get('val_composite_formula', '')}` — {vr.get('note', '')}"
        )
    )

if bench_df.empty:
    print("No benchmark table available for this run.")
else:
    sort_cols = [c for c in ("stage", "val_composite", "val_mae") if c in bench_df.columns]
    display(bench_df.sort_values(sort_cols, na_position="last"))

    win = (
        bench_df[bench_df["selected"] == "yes"]
        if "selected" in bench_df.columns
        else pd.DataFrame()
    )
    if not win.empty:
        w = win.iloc[0]
        display(
            Markdown(
                f"**Final choice (this run):** `{w.get('model')}` with **target = `{w.get('target_strategy')}`** "
                f"(best **val_composite** among tuned finalists on the calendar validation day). "
                f"Holdout metrics use the **latest** day only, so they can differ from validation when the listing mix changes."
            )
        )

if metrics.get("model_comparison"):
    display(Markdown("**Selection rule (from metrics):**"))
    display(Markdown(f"> {metrics['model_comparison'].get('selection_rule', '')}"))

if metrics.get("benchmark", {}).get("skip_reason"):
    display(
        Markdown(
            f"_Note: benchmark skipped or limited — `{metrics['benchmark']['skip_reason']}`._"
        )
    )


### Interpretation (model comparison)

- **val_composite** (see table) combines **MAE** with penalties for heavy tails (**RMSE**) and systematic shift (**bias**). That favors configurations that are not just slightly better on average but also less volatile and less one-sided on the validation day.
- **MAPE** is displayed for context but is **not** the ranking metric: with many inexpensive listings, small dollar errors still inflate MAPE.
- **Raw vs log:** both are benchmarked; the winner is whichever **(model, target)** pair scores best on **val_composite** in **dollars** after **expm1** for log models.
- **Why forests vs boosting:** the table shows which inductive bias fit this week's validation slice; the pipeline records the decision instead of hard-coding one learner.


## 4. Final holdout performance (latest run)

These values come from **`metrics.json`** for the latest synced run (same numbers as **`metrics.csv`**).

In [4]:
if metrics:
    hold = pd.DataFrame(
        [
            {
                "eval_date_local": metrics.get("eval_date_local"),
                "train_rows": metrics.get("train_rows"),
                "holdout_rows": metrics.get("holdout_rows"),
                "MAE": metrics.get("mae"),
                "MAPE_%": metrics.get("mape"),
                "RMSE": metrics.get("rmse"),
                "Bias": metrics.get("bias"),
                "chosen_model": metrics.get("chosen_model"),
                "target_strategy": metrics.get("target_strategy"),
                "target_transform": metrics.get("target_transform"),
            }
        ]
    )
    display(hold)
else:
    print("No metrics.json loaded.")

,eval_date_local,train_rows,holdout_rows,MAE,MAPE_%,RMSE,Bias,chosen_model,target_strategy,target_transform
0,2026-04-04,209,270,6187.672324,136.239278,10679.883124,3204.186433,HistGradientBoostingRegressor,raw,price_num raw USD


### Row filtering (latest run)

Counts come from **`metrics.json` → `filtering`** (same rules as training).

In [5]:
fl = metrics.get("filtering") or {}
if fl:
    fd = pd.DataFrame([{"metric": k, "value": v} for k, v in fl.items()])
    display(fd)
else:
    print("No filtering block in metrics.json.")

,metric,value
0,input_rows,526.00
1,dropped_missing_or_nonpositive_price,16.00
2,dropped_extreme_price,2.00
3,dropped_bad_year,20.00
4,dropped_bad_mileage,3.00
5,dropped_title_status_nonstandard,4.00
6,dropped_condition_project,2.00
7,output_rows,479.00
8,rows_removed_total,47.00
9,pct_rows_removed,8.94


## 5. Performance trends across training runs

**`results/history/metrics_history.csv`** appends one row per successful `train-dt` job (after sync). Prefer **`timestamp_utc`** on the x-axis when present; otherwise **`eval_date_local`** or **`run_id`**.

Duplicate `run_id` rows can appear if the same run was synced twice -- treat the file as operational telemetry, not a clean experiment log.


In [6]:
hist = pd.DataFrame()
hist_path = RESULTS / "history" / "metrics_history.csv"

if not RESULTS.is_dir():
    print("No results/ folder.")
elif not hist_path.is_file():
    print("No metrics_history.csv yet.")
else:
    try:
        hist = pd.read_csv(hist_path)
        if "timestamp_utc" in hist.columns:
            hist["timestamp_utc"] = pd.to_datetime(
                hist["timestamp_utc"], utc=True, errors="coerce"
            )
        if "eval_date_local" in hist.columns:
            hist["eval_date_local"] = pd.to_datetime(
                hist["eval_date_local"], errors="coerce"
            ).dt.date.astype(str)
        display(hist.tail(10))
    except Exception as e:
        print(f"Could not load history: {e}")

if hist.empty:
    print("Skip trend plots.")
else:
    if "timestamp_utc" in hist.columns and hist["timestamp_utc"].notna().any():
        x = hist["timestamp_utc"]
        xlabel = "timestamp_utc"
    elif "eval_date_local" in hist.columns:
        x = hist["eval_date_local"].astype(str)
        xlabel = "eval_date_local"
    elif "run_id" in hist.columns:
        x = hist["run_id"].astype(str)
        xlabel = "run_id"
    else:
        x = range(len(hist))
        xlabel = "row index"

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    for ax, (col, title) in zip(
        axes.ravel(),
        [
            ("mae", "MAE (USD)"),
            ("mape", "MAPE (%)"),
            ("rmse", "RMSE (USD)"),
            ("bias", "Bias (mean pred - actual)"),
        ],
    ):
        if col in hist.columns:
            ax.plot(x, hist[col].astype(float), marker="o", markersize=4, alpha=0.85)
            ax.set_title(title)
            ax.set_xlabel(xlabel)
            ax.tick_params(axis="x", rotation=35)
            ax.grid(True, alpha=0.3)
        else:
            ax.set_visible(False)
    plt.tight_layout()
    plt.show()


### Interpretation (trends)

- **MAE / RMSE** moves reflect both model changes **and** a new **holdout day** (different cars, prices, and seller mix). A spike on one date is not proof the model "broke" -- check validation vs holdout in `metrics.json` for that `run_id`.
- **MAPE** is especially sensitive to **low-priced** listings (small denominators). Read it next to **MAE** and **bias**.
- **Bias** (mean predicted minus actual on the holdout): sustained positive bias means listings on that day were **underpriced** by the model on average; negative bias means **overpriced** on average, relative to realized ask prices in the scrape.


## 6. Permutation importance (latest run)

Holdout **permutation importance** measures how much **dollar MAE** worsens when a feature is shuffled (conditional on the fitted pipeline). Higher **importance_mean** ⇒ the model relied more on that input on the holdout slice.

In [7]:
import glob

imp_path = None
if run_id:
    cand = RESULTS / "permutation_importance" / f"{run_id}-permutation_importance.csv"
    if cand.is_file():
        imp_path = cand

if imp_path is None and RESULTS.is_dir():
    all_imp = sorted(
        glob.glob(str(RESULTS / "permutation_importance" / "*-permutation_importance.csv"))
    )
    if all_imp:
        imp_path = Path(all_imp[-1])

if imp_path is None or not imp_path.is_file():
    print("No permutation importance CSV found.")
else:
    try:
        dfi = pd.read_csv(imp_path)
        need = {"feature", "importance_mean"}
        if not need.issubset(dfi.columns):
            print("Unexpected columns in permutation file:", list(dfi.columns))
        else:
            top = dfi.sort_values("importance_mean", ascending=True).tail(15)
            fig, ax = plt.subplots(figsize=(11, 7))
            ax.barh(top["feature"], top["importance_mean"])
            ax.set_title(f"Permutation importance (top 15) — {imp_path.name}")
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f"Could not plot importance: {e}")


### Interpretation (importance)

- **`vehicle_age`**, **`log_mileage`**, **`year_num`**, or **`mileage_num`** often rank high: price moves with depreciation and odometer in this feature encoding.
- **Make / model / location** buckets capture segment-level price levels; importance is **marginal to the fitted model**, not a causal claim.
- Shuffled-feature **dollar MAE** worsens more for features the pipeline relied on to fit the holdout slice.


## 7. Partial dependence plots (latest run, top 3)

PDPs show **average predicted price** vs one feature, **marginalizing** over the others (holdout rows from the training job). They are **directional**, not causal; sparse categories or few rows can look jagged.

Only the **three** features with highest holdout permutation importance are exported as PNGs for this run.


In [8]:
pdp_files = []
if run_id and (RESULTS / "pdp").is_dir():
    pdp_files = sorted((RESULTS / "pdp").glob(f"{run_id}_*.png"))

if not pdp_files and (RESULTS / "pdp").is_dir():
    pdp_files = sorted((RESULTS / "pdp").glob("*.png"))[-3:]

if not pdp_files:
    print("No PDP PNGs in results/pdp/ for this run (train + sync after PDP export).")
elif Image is None:
    print("IPython.display.Image not available; paths:", pdp_files)
else:
    for fp in pdp_files[:3]:
        display(Markdown(f"**{fp.name}**"))
        display(Image(filename=str(fp), width=900))


### Interpretation (PDPs)

- **Slope:** steeper regions mean higher average predicted price along that feature range, holding the encoding fixed.
- **Flat or wiggly regions:** few rows, weak signal, or strong interactions not visible in 1-D.
- PDPs use the **holdout** rows seen by `train-dt` for that run; match them to the permutation-importance ordering above.


## 8. Discussion (honest read)

- **What improved:** the pipeline compares **four** regressors, **raw vs log** targets, and **tuned** finalists on a **calendar validation** day, then evaluates on a **strict forward** holdout. Artifacts (`model_benchmark.csv`, importance, PDPs) make the choice auditable.
- **What stays hard:** Craigslist asks are noisy; one **calendar day** of holdout can be an easy or brutal slice. **MAPE** will swing when the mix includes many cheap cars.
- **Bias:** read alongside MAE -- it describes systematic shift on that holdout day, not "accuracy" alone.
- **Compared to a single default model:** this workflow documents **why** a specific **(model, target)** pair was selected for each run (via **val_composite** on validation, then holdout metrics for reporting).


## 9. Final reflection / grader checklist

- **Repro:** train with `dry_run: false`, run **Sync model artifacts**, open this notebook from repo root (or Colab with `NOTEBOOK_REPO_URL`).
- **Read first:** `model_benchmark.csv` + **Final choice** callout above, then holdout table, then trends.
- **Next steps (if this were production):** rolling weekly holdout summaries, optional region stratification, and text/trim features only if leakage and time alignment are controlled -- out of scope for this midterm.


---

**Metadata:** For the latest run, `model_info.json` lists **`feature_notes`**, **`filtering`**, and **`benchmark.feature_variant`** (numeric feature set A / B / C) used when the Cloud Function trained.
